# Naija-Speech — STT Vertical Slice (Unsloth)

**What this notebook does**, end to end, on a *free* Google Colab **T4 GPU**:

1. **Zero-shot baseline** — run pretrained Whisper on the Nigerian test set and measure
   WER/CER per macro-accent. This is the **number fine-tuning must beat**.
2. **Fine-tune** — LoRA-adapt `whisper-large-v3-turbo` on the curated corpus using
   **Unsloth** (2x faster; 4-bit makes an 809M model fit a T4 — no A100 rental).
3. **Evaluate** — run the fine-tuned model on the same test set and print the
   **before → after** WER, per accent. These are your **Chapter 5** numbers.

> **Plain English:** *WER (Word Error Rate)* = the % of words the model gets wrong. Lower is
> better. *Zero-shot* = the model as-shipped, no training on our data. *LoRA* = train ~1% of the
> weights (cheap) instead of all of them. We measure the gap, then close it.

**You need:** a **T4 GPU runtime** (Runtime → Change runtime type → T4 GPU). A W&B key is
optional (logs the metrics). The curated dataset is public, so an HF token is optional.

## 0 — Get the code and install Unsloth

`unsloth` brings its own compatible `transformers` / `peft` / `accelerate` / `bitsandbytes`
stack, so we install it **after** cloning. If the install misbehaves, do
**Runtime → Restart session** once and re-run this cell (a one-time Colab quirk).

In [ ]:
import os, subprocess

# Fresh Colab clones the repo; re-runs just pull the latest pushed code.
if not os.path.exists("scripts/03_finetune_whisper_lora.py"):
    if not os.path.exists("naija-speech"):
        subprocess.run(
            ["git", "clone", "https://github.com/Johniblazee/naija-speech.git"],
            check=True,
        )
    os.chdir("naija-speech")
subprocess.run(["git", "pull", "--ff-only"], check=False)
print("Working dir:", os.getcwd())

In [ ]:
# Confirm a GPU is attached BEFORE the long installs.
import torch
assert torch.cuda.is_available(), \
    "No GPU! Set Runtime -> Change runtime type -> T4 GPU, then re-run."
print("GPU:", torch.cuda.get_device_name(0))

# Unsloth (2x faster Whisper training + 4-bit). Pulls a compatible ML stack.
%pip install -q unsloth
%pip install -q jiwer      # WER/CER metric used by src/metrics.py

## 1 — Authenticate

- **W&B key** (optional but recommended): logs the WER/CER so your Chapter 5 curves live in one
  place. Press **Enter** to skip.
- **Hugging Face token** (optional): the curated dataset is **public**, so you can skip it —
  only needed if you later make the dataset private.

In [ ]:
import os, getpass
from huggingface_hub import login

# W&B — optional. Press Enter to skip.
if not os.environ.get("WANDB_API_KEY"):
    wb = getpass.getpass("W&B API key (optional, Enter to skip): ")
    if wb:
        os.environ["WANDB_API_KEY"] = wb

# HF — optional (public dataset). Press Enter to skip.
if not os.environ.get("HF_TOKEN"):
    tok = getpass.getpass("Hugging Face token (optional, Enter to skip): ")
    if tok:
        os.environ["HF_TOKEN"] = tok
        login(tok)

## 2 — Pick the run size

One knob for the whole slice:

- **`QUICK = True`** → fast sanity pass on a **subset** (a first, rough number in ~20 min). Good
  for checking the whole flow works and eyeballing that fine-tuning helps.
- **`QUICK = False`** → the **full** thesis run: zero-shot on all 4,583 test clips, fine-tune one
  epoch over the 40,713-clip train split, evaluate on the full test set. Takes a few hours — the
  fine-tune **checkpoints** so a Colab disconnect can resume (see the note in Section 4).

In [ ]:
QUICK = True   # True = fast subset pass; False = full thesis run

MODEL_CONFIG   = "configs/stt_whisper_large_v3_turbo_unsloth.yaml"
BASELINE_LIMIT = 300  if QUICK else None   # zero-shot test clips (None = full 4,583)
FT_MAX_TRAIN   = 2000 if QUICK else None   # fine-tune train clips (None = full 40,713)
FT_MAX_STEPS   = 200  if QUICK else -1     # -1 = one full epoch
EVAL_LIMIT     = 300  if QUICK else None   # eval test clips (None = full)

print("Mode:", "QUICK (subset)" if QUICK else "FULL (thesis run)")

## 3 — Smoke test the Unsloth path (fail fast)

Before the real run, prove the whole loop works on your T4 in ~3 minutes: load
`whisper-large-v3-turbo` in **4-bit**, attach LoRA, map 300 clips, run **20 steps**, save the
adapter. If this cell finishes with *"Saved LoRA adapter"*, the setup is good. It's throwaway.

In [ ]:
!python scripts/03_finetune_whisper_lora.py \
    --model-config "$MODEL_CONFIG" \
    --backend unsloth --max-train 300 --max-steps 20

## 4 — Zero-shot baseline (the number to beat)

Runs the **pretrained** model on the Nigerian test set and reports WER/CER overall and per
macro-accent. Writes `outputs/zeroshot/baseline_results.csv`.

> The `Other` accent bucket is the largest and most mixed; expect Yoruba/Igbo/Hausa to differ —
> that spread is exactly the **fairness** story your thesis measures.

In [ ]:
limit = f"--limit {BASELINE_LIMIT}" if BASELINE_LIMIT else ""
!python scripts/02_zeroshot_baseline.py --model-config "$MODEL_CONFIG" {limit}

## 5 — Fine-tune with LoRA (Unsloth)

LoRA-adapts the model on the curated corpus. Saves the adapter to
`outputs/whisper_turbo_unsloth/adapter`.

> **Colab disconnected mid-run?** Just re-run this cell with `--resume` appended (it's added
> automatically below when the checkpoint folder exists). It picks up from the last saved
> checkpoint — no work is lost. Free Colab sessions are time-limited, so for `QUICK = False`
> expect this to possibly span a session or two.

> **CUDA out of memory?** Edit `configs/stt_whisper_large_v3_turbo_unsloth.yaml`: drop
> `per_device_train_batch_size` to `2` (or `1`) and raise `gradient_accumulation_steps` to keep
> the effective batch near 16.

In [ ]:
import glob, os
mt = f"--max-train {FT_MAX_TRAIN}" if FT_MAX_TRAIN else ""
ms = f"--max-steps {FT_MAX_STEPS}"  # -1 = full epoch; positive = step budget
# Auto-resume if a checkpoint already exists (e.g. after a disconnect).
resume = "--resume" if glob.glob("outputs/whisper_turbo_unsloth/checkpoint-*") else ""

!python scripts/03_finetune_whisper_lora.py \
    --model-config "$MODEL_CONFIG" --backend unsloth {mt} {ms} {resume}

## 6 — Evaluate + before/after table

Runs the **fine-tuned** model on the same test set and writes
`outputs/whisper_turbo_unsloth/results/finetuned_results.csv`, then prints the zero-shot vs
fine-tuned WER side by side. **These are your Chapter 5 numbers — copy the table.**

In [ ]:
limit = f"--limit {EVAL_LIMIT}" if EVAL_LIMIT else ""
!python scripts/04_evaluate.py --model-config "$MODEL_CONFIG" {limit}

In [ ]:
# Merge the two result CSVs into one before/after view.
import pandas as pd, os

base = pd.read_csv("outputs/zeroshot/baseline_results.csv")
ft   = pd.read_csv("outputs/whisper_turbo_unsloth/results/finetuned_results.csv")

m = base.merge(ft, on=["group", "key"], suffixes=("_zeroshot", "_finetuned"))
m["WER_delta"] = (m["wer_finetuned"] - m["wer_zeroshot"]).round(3)
m["rel_improvement_%"] = ((1 - m["wer_finetuned"] / m["wer_zeroshot"]) * 100).round(1)
cols = ["group", "key", "n_zeroshot", "wer_zeroshot", "wer_finetuned",
        "WER_delta", "rel_improvement_%"]
view = m[cols].sort_values(["group", "wer_zeroshot"], ascending=[True, False])
print("Zero-shot vs fine-tuned WER (lower = better; negative delta = improvement):\n")
print(view.to_string(index=False))
view

## What's next

- If `QUICK = True` gave a sensible improvement, set **`QUICK = False`** and re-run Sections 4–6
  for the full thesis numbers.
- Run `scripts/05_verify_corpus.py` once to generate the per-accent hours/speaker tables for
  Chapter 5.
- Paste the before/after table back into the chat and it goes straight into **Chapter 5**.
- Then the **TTS track** (StyleTTS 2 first).

Progress is logged in `docs/PROGRESS.md`.